# Lesson 5.2 — Data Sensitivity, Privacy, and Regulation

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Recognize regulated and sensitive data types
- Apply anonymization/redaction before using sensitive data
- Follow privacy and regulatory considerations in daily work

**You will produce:** A simple redaction/anonymization routine and a saved data-handling rule

## Concepts

Protecting data is a core responsible-use skill. **Sensitive/regulated data** includes personal identifiers (names, account numbers, SSNs, health info), confidential business data, and anything covered by law or contract (e.g., GDPR, HIPAA).

**Key exam insight (Sample Question 3):** When someone wants to upload a spreadsheet of customer **names and account numbers** and policy restricts regulated personal data, the right action is to **remove or anonymize the identifiers before uploading**, consistent with policy. Uploading as-is violates policy; telling Claude 'don't retain it' does **not** satisfy the control; and skipping the analysis is unnecessary because anonymization lets it proceed.

Practical rule: **minimize and anonymize** - share only what's needed, strip identifiers first.

This maps to Domain 6: *Apply data sensitivity, regulatory, and privacy considerations*.

> **Reference:** [Anthropic - Privacy & data handling](https://privacy.anthropic.com)

In [1]:
# !pip install anthropic   # run once if the SDK is not installed
# %pip install python-dotenv  # only if it isn't installed
import os
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(".env", usecwd=True)

if not env_path:
    raise FileNotFoundError("Could not find .env from the notebook's working directory")

load_dotenv(env_path)
print("API key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


def ask_claude(prompt, system=None, model="claude-sonnet-5-5", max_tokens=600):
    """Send a prompt to Claude. Falls back to a printed mock if no API key is set,
    so this notebook always runs. Set ANTHROPIC_API_KEY in your environment to use the real API.
    NOTE: Verify current model names at https://docs.anthropic.com (they change over time)."""
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("[No ANTHROPIC_API_KEY found - showing a MOCK response so the lesson still runs.]\n")
        return "[MOCK CLAUDE OUTPUT]\nThis is a placeholder. With a real key, Claude's answer appears here."
    try:
        import anthropic
        client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
        kwargs = {"model": model, "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        # Newer models may return thinking blocks first; keep only the text blocks
        return "".join(b.text for b in resp.content if b.type == "text")
    except Exception as e:
        print(f"[Could not reach the API: {e}. Showing a mock response.]\n")
        return "[MOCK CLAUDE OUTPUT] Placeholder response."

print("Helper ready. Call ask_claude('your prompt here').")

API key loaded: True
Helper ready. Call ask_claude('your prompt here').


## Code-Along: Step 1 — A tiny anonymization helper

Redact common identifiers before any upload. (Illustrative - always confirm with your real policy/tools.)

In [2]:
import re
def anonymize(text):
    text = re.sub(r'\b\d{6,}\b', '[ACCOUNT]', text)            # long numbers -> account ids
    text = re.sub(r'\b[\w.]+@[\w.]+\b', '[EMAIL]', text)        # emails
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)      # ssn-like
    return text

sample = 'Customer Jane Doe, account 100245678, email jane@acme.com, SSN 123-45-6789.'
print('BEFORE:', sample)
print('AFTER :', anonymize(sample))
print('\nNote: names may still need manual removal - this is a starting point, not a guarantee.')

BEFORE: Customer Jane Doe, account 100245678, email jane@acme.com, SSN 123-45-6789.
AFTER : Customer Jane Doe, account [ACCOUNT], email [EMAIL], SSN [SSN].

Note: names may still need manual removal - this is a starting point, not a guarantee.


## Exercise

Anonymize a realistic (fake) record and save your data-handling rule. Replace the TODO sample with fake data - never paste real regulated data.

In [3]:
my_sample = 'SLDC 123-45-6789 john.doe@example.com'
cleaned = anonymize(my_sample)
rule = ('# My data-handling rule\n\n'
        '1. Never upload regulated personal data as-is.\n'
        '2. Remove or anonymize identifiers FIRST (names, account numbers, emails, SSNs).\n'
        '3. Share only the minimum data needed.\n'
        '4. Follow org policy; when unsure, ask/escalate.\n\n'
        f'Example cleaned record:\n{cleaned}\n')
with open('data_handling_rule.md','w',encoding='utf-8') as f:
    f.write(rule)
print('Saved data_handling_rule.md')
print(rule)

Saved data_handling_rule.md
# My data-handling rule

1. Never upload regulated personal data as-is.
2. Remove or anonymize identifiers FIRST (names, account numbers, emails, SSNs).
3. Share only the minimum data needed.
4. Follow org policy; when unsure, ask/escalate.

Example cleaned record:
SLDC [SSN] [EMAIL]



## Key Takeaways

- Anonymize/redact regulated identifiers BEFORE uploading (Sample Q3)
- 'Don't retain it' is not a valid substitute for the policy control
- Minimize data - share only what's needed
- Follow privacy/regulatory rules (GDPR, HIPAA, org policy)

---

**Next:** [Lesson 5.3 — Organizational AI Policy and Ethics](./lesson-5.3-org-policy-and-ethics.ipynb)